# 17. Final Model Pipeline and Inference

## Objective
Build a final production-ready inference pipeline that:

1. Uses a balanced threshold (not naive F1-only)
2. Applies regime-aware and confidence-based routing
3. Produces realistic predictions with reduced over-prediction bias

Rules preserved: no leakage, no shuffle, same temporal split boundaries as prior notebooks.

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple
import json
import logging
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 220)

RANDOM_STATE = 42
TRAIN_START = pd.Timestamp('2023-04-18')
TRAIN_END = pd.Timestamp('2024-12-31')
TEST_START = pd.Timestamp('2025-01-01')
TEST_END = pd.Timestamp('2025-12-30')
SEQUENCE_WINDOW = 20

UNCERTAIN_LOW = 0.48
UNCERTAIN_HIGH = 0.52

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

logger = logging.getLogger('final_pipeline')
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(h)
logger.setLevel(logging.INFO)
logger.info('Using device: %s', DEVICE)

[INFO] Using device: cpu


## STEP 1: Load Data + Models

In [3]:
def resolve_project_root() -> Path:
    cwd = Path.cwd()
    for c in [cwd, cwd.parent, cwd.parent.parent]:
        if (c / 'ml_pipeline').exists() and (c / 'CLAUDE.md').exists():
            return c
    raise FileNotFoundError('Unable to locate project root with ml_pipeline and CLAUDE.md')


def resolve_paths(root: Path) -> Dict[str, Path]:
    base = root / 'ml_pipeline'
    return {
        'dataset': base / 'Market_Data' / 'processed' / 'final_model_dataset_with_volatility.parquet',
        'baseline_metrics': base / 'models' / 'baseline_metrics_summary.json',
        'dl_metrics': base / 'models' / 'dl_metrics_summary.json',
        'rf_model': base / 'models' / 'rf_baseline.pkl',
        'xgb_model': base / 'models' / 'xgb_baseline.pkl',
        'lstm_model': base / 'models' / 'lstm_model.pt',
        'transformer_model': base / 'models' / 'transformer_model.pt',
        'optimized_threshold': base / 'models' / 'optimized_threshold.json',
        'ensemble_config': base / 'models' / 'ensemble_config.json',
        'final_predictions': base / 'Market_Data' / 'final' / 'final_predictions.parquet',
    }


ROOT = resolve_project_root()
PATHS = resolve_paths(ROOT)

required = ['dataset', 'baseline_metrics', 'dl_metrics', 'rf_model', 'xgb_model', 'lstm_model', 'transformer_model']
for k in required:
    assert PATHS[k].exists(), f'Missing required artifact: {PATHS[k]}'

df = pd.read_parquet(PATHS['dataset']).copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

assert {'Date', 'Ticker', 'target', 'volatility_regime_label'}.issubset(df.columns)
assert df['Date'].min() >= pd.Timestamp('2023-01-01')
assert df['Date'].max() <= pd.Timestamp('2026-01-01')

baseline_metrics = json.loads(PATHS['baseline_metrics'].read_text(encoding='utf-8'))
dl_metrics = json.loads(PATHS['dl_metrics'].read_text(encoding='utf-8'))

with open(PATHS['rf_model'], 'rb') as f:
    rf_model = pickle.load(f)
with open(PATHS['xgb_model'], 'rb') as f:
    xgb_model = pickle.load(f)

logger.info('Loaded dataset shape: %s', df.shape)
display(df[['Date', 'Ticker', 'target', 'volatility_regime_label']].head())

[INFO] Loaded dataset shape: (63541, 168)


,Date,Ticker,target,volatility_regime_label
0,2023-04-18,ABB,0,MEDIUM
1,2023-04-19,ABB,0,MEDIUM
2,2023-04-20,ABB,0,MEDIUM
3,2023-04-21,ABB,1,MEDIUM
4,2023-04-24,ABB,1,LOW


## STEP 2: Load Optimized Config (threshold and ensemble weights)

In [4]:
threshold_cfg = {}
ensemble_cfg = {'transformer_weight': 0.7, 'xgb_weight': 0.3}

if PATHS['optimized_threshold'].exists():
    threshold_cfg = json.loads(PATHS['optimized_threshold'].read_text(encoding='utf-8'))
if PATHS['ensemble_config'].exists():
    ensemble_cfg = json.loads(PATHS['ensemble_config'].read_text(encoding='utf-8'))

prior_threshold = float(threshold_cfg.get('optimal_threshold', ensemble_cfg.get('threshold', 0.5)))
w_transformer = float(ensemble_cfg.get('transformer_weight', 0.7))
w_xgb = float(ensemble_cfg.get('xgb_weight', 1.0 - w_transformer))

assert 0 <= prior_threshold <= 1
assert abs((w_transformer + w_xgb) - 1.0) < 1e-8

logger.info('Loaded prior threshold: %.3f', prior_threshold)
logger.info('Loaded weights: transformer=%.2f, xgb=%.2f', w_transformer, w_xgb)

[INFO] Loaded prior threshold: 0.410
[INFO] Loaded weights: transformer=0.70, xgb=0.30


## Helper: Rebuild Step-15/16 Consistent Predictions

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :])).squeeze(-1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pe, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TransformerClassifier(nn.Module):
    def __init__(self, input_size: int, d_model: int = 64, n_heads: int = 4, num_layers: int = 2, dropout: float = 0.2, max_len: int = 20):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True,
            dim_feedforward=d_model * 4,
            activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)
        return self.head(self.dropout(h[:, -1, :])).squeeze(-1)


def select_features(data: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    drop_cols = ['Date', 'Ticker', 'target', 'volatility_regime_label', 'vol_cluster_regime_name']
    feature_cols = [c for c in data.columns if c not in drop_cols]
    X = data[feature_cols].copy()
    y = data['target'].astype(int).copy()
    assert not X.select_dtypes(exclude=[np.number]).columns.tolist(), 'Non-numeric features found.'
    assert X.isna().sum().sum() == 0, 'NaNs in model features.'
    return X, y, feature_cols


def split_data(data: pd.DataFrame, X: pd.DataFrame, y: pd.Series):
    train_mask = (data['Date'] >= TRAIN_START) & (data['Date'] <= TRAIN_END)
    test_mask = (data['Date'] >= TEST_START) & (data['Date'] <= TEST_END)

    X_train, X_test = X.loc[train_mask].copy(), X.loc[test_mask].copy()
    y_train, y_test = y.loc[train_mask].copy(), y.loc[test_mask].copy()
    df_train, df_test = data.loc[train_mask].copy(), data.loc[test_mask].copy()
    assert df_train['Date'].max() < df_test['Date'].min(), 'Temporal overlap detected.'
    return X_train, X_test, y_train, y_test, df_train, df_test


def scale_features_train_only(X_train: pd.DataFrame, X_test: pd.DataFrame):
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_scaled, X_test_scaled


def create_sequences_with_meta(df_part: pd.DataFrame, X_scaled_part: pd.DataFrame, y_part: pd.Series, feature_cols: List[str], window: int = 20):
    seq_X, seq_y, meta_rows = [], [], []
    meta = df_part[['Ticker', 'Date', 'volatility_regime_label']].reset_index(drop=True)
    feat = X_scaled_part[feature_cols].reset_index(drop=True)
    tmp = pd.concat([meta, feat], axis=1)
    tmp['target'] = y_part.reset_index(drop=True).astype(int)
    tmp = tmp.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    for ticker, grp in tmp.groupby('Ticker', sort=False):
        values = grp[feature_cols].to_numpy(dtype=np.float32)
        labels = grp['target'].to_numpy(dtype=np.int64)
        dates = grp['Date'].to_numpy()
        regimes = grp['volatility_regime_label'].astype(str).to_numpy()
        if len(grp) <= window:
            continue
        for i in range(window, len(grp)):
            seq_X.append(values[i - window:i])
            seq_y.append(labels[i])
            meta_rows.append({'Ticker': ticker, 'Date': pd.Timestamp(dates[i]), 'Regime': regimes[i], 'target': int(labels[i])})
    return np.asarray(seq_X), np.asarray(seq_y), pd.DataFrame(meta_rows)


def predict_torch_binary(model: nn.Module, X_np: np.ndarray, batch_size: int = 512):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.tensor(X_np[i:i + batch_size], dtype=torch.float32, device=DEVICE)
            logits = model(xb)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    prob = np.concatenate(probs)
    pred = (prob >= 0.5).astype(int)
    return pred, prob


X_all, y_all, feature_cols = select_features(df)
X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, X_all, y_all)
X_train_scaled, X_test_scaled = scale_features_train_only(X_train, X_test)

base_train = df_train[['Date', 'Ticker', 'volatility_regime_label']].copy().rename(columns={'volatility_regime_label': 'Regime'})
base_train['target'] = y_train.values
base_train['XGBoost_prob'] = xgb_model.predict_proba(X_train)[:, 1]
base_train['XGBoost_pred'] = (base_train['XGBoost_prob'] >= 0.5).astype(int)

base_test = df_test[['Date', 'Ticker', 'volatility_regime_label']].copy().rename(columns={'volatility_regime_label': 'Regime'})
base_test['target'] = y_test.values
base_test['XGBoost_prob'] = xgb_model.predict_proba(X_test)[:, 1]
base_test['XGBoost_pred'] = (base_test['XGBoost_prob'] >= 0.5).astype(int)

X_seq_train, _, seq_meta_train = create_sequences_with_meta(df_train, X_train_scaled, y_train, feature_cols, window=SEQUENCE_WINDOW)
X_seq_test, _, seq_meta_test = create_sequences_with_meta(df_test, X_test_scaled, y_test, feature_cols, window=SEQUENCE_WINDOW)

transformer_model = TransformerClassifier(input_size=len(feature_cols), max_len=SEQUENCE_WINDOW).to(DEVICE)
transformer_model.load_state_dict(torch.load(PATHS['transformer_model'], map_location=DEVICE))

trf_train_pred, trf_train_prob = predict_torch_binary(transformer_model, X_seq_train)
trf_test_pred, trf_test_prob = predict_torch_binary(transformer_model, X_seq_test)

seq_train = seq_meta_train.copy()
seq_train['Transformer_pred'] = trf_train_pred
seq_train['Transformer_prob'] = trf_train_prob

seq_test = seq_meta_test.copy()
seq_test['Transformer_pred'] = trf_test_pred
seq_test['Transformer_prob'] = trf_test_prob

aligned_train = seq_train.merge(base_train[['Date', 'Ticker', 'XGBoost_prob', 'XGBoost_pred']], on=['Date', 'Ticker'], how='left')
aligned_test = seq_test.merge(base_test[['Date', 'Ticker', 'XGBoost_prob', 'XGBoost_pred']], on=['Date', 'Ticker'], how='left')

assert aligned_train['XGBoost_prob'].isna().sum() == 0
assert aligned_test['XGBoost_prob'].isna().sum() == 0
assert aligned_train['Date'].max() < aligned_test['Date'].min()

logger.info('Aligned rows -> train: %s | test: %s', f'{len(aligned_train):,}', f'{len(aligned_test):,}')
display(aligned_test.head())

[INFO] Aligned rows -> train: 37,813 | test: 21,888


,Ticker,Date,Regime,target,Transformer_pred,Transformer_prob,XGBoost_prob,XGBoost_pred
0,ABB,2025-01-29,HIGH,0,0,0.494498,0.529273,1
1,ABB,2025-01-30,HIGH,1,1,0.502987,0.691457,1
2,ABB,2025-01-31,HIGH,0,0,0.492936,0.351550,0
3,ABB,2025-02-01,HIGH,0,1,0.506073,0.658980,1
4,ABB,2025-02-03,HIGH,1,0,0.496291,0.591766,1


## STEP 3: Balanced Threshold Fix

In [6]:
def build_validation_mask(df_part: pd.DataFrame, val_fraction: float = 0.2):
    dates = np.sort(df_part['Date'].drop_duplicates().to_numpy())
    split_idx = max(1, int(len(dates) * (1 - val_fraction)))
    split_idx = min(split_idx, len(dates) - 1)
    val_start = pd.Timestamp(dates[split_idx])
    val_mask = df_part['Date'] >= val_start
    train_mask = ~val_mask
    assert df_part.loc[train_mask, 'Date'].max() < df_part.loc[val_mask, 'Date'].min()
    return train_mask, val_mask, val_start


def ensemble_prob(transformer_prob: np.ndarray, xgb_prob: np.ndarray, wt: float, wx: float) -> np.ndarray:
    return (wt * transformer_prob) + (wx * xgb_prob)


def routed_prob(regimes: pd.Series, transformer_prob: np.ndarray, xgb_prob: np.ndarray, wt: float, wx: float) -> np.ndarray:
    ens = ensemble_prob(transformer_prob, xgb_prob, wt=wt, wx=wx)
    r = regimes.astype(str).str.upper()
    return np.where(r == 'HIGH', transformer_prob, np.where(r == 'LOW', xgb_prob, ens))


def optimize_balanced_threshold(y_true: np.ndarray, y_prob: np.ndarray, precision_floor: float):
    thresholds = np.round(np.arange(0.30, 0.701, 0.01), 2)
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)
        combo = (0.6 * f1) + (0.4 * p)
        rows.append({'threshold': float(t), 'Precision': p, 'Recall': r, 'F1': f1, 'Combined': combo, 'meets_precision_floor': p >= precision_floor})
    table = pd.DataFrame(rows)
    eligible = table[table['meets_precision_floor']]
    best = (eligible if len(eligible) > 0 else table).sort_values(['Combined', 'F1', 'Precision'], ascending=False).iloc[0]
    return float(best['threshold']), table


_, val_mask, val_start = build_validation_mask(aligned_train, val_fraction=0.2)
val_df = aligned_train.loc[val_mask].copy()

val_df['routed_prob'] = routed_prob(
    regimes=val_df['Regime'],
    transformer_prob=val_df['Transformer_prob'].to_numpy(),
    xgb_prob=val_df['XGBoost_prob'].to_numpy(),
    wt=w_transformer,
    wx=w_xgb,
)

precision_floor = float(dl_metrics.get('Transformer', {}).get('Precision', 0.50))
balanced_threshold, threshold_table = optimize_balanced_threshold(
    y_true=val_df['target'].to_numpy(),
    y_prob=val_df['routed_prob'].to_numpy(),
    precision_floor=precision_floor,
)

logger.info('Validation start date: %s', val_start.date())
logger.info('Prior threshold: %.3f | Balanced threshold: %.3f', prior_threshold, balanced_threshold)
logger.info('Precision floor: %.4f', precision_floor)

display(threshold_table.sort_values(['Combined', 'F1'], ascending=False).head(10).round(4))

[INFO] Validation start date: 2024-09-04
[INFO] Prior threshold: 0.410 | Balanced threshold: 0.500
[INFO] Precision floor: 0.5085


,threshold,Precision,Recall,F1,Combined,meets_precision_floor
20,0.50,0.6757,0.7462,0.7092,0.6958,True
19,0.49,0.6419,0.8306,0.7242,0.6912,True
18,0.48,0.6252,0.8724,0.7284,0.6871,True
22,0.52,0.7994,0.4954,0.6117,0.6868,True
21,0.51,0.7431,0.5699,0.6451,0.6843,True
23,0.53,0.8303,0.4533,0.5864,0.6840,True
17,0.47,0.6116,0.8948,0.7266,0.6806,True
16,0.46,0.6040,0.9109,0.7264,0.6774,True
24,0.54,0.8485,0.4194,0.5613,0.6762,True
15,0.45,0.5948,0.9249,0.7240,0.6723,True


## STEP 4: Final Prediction Logic + STEP 5: Confidence Filter

In [7]:
def apply_final_logic(df_pred: pd.DataFrame, threshold: float, wt: float, wx: float, uncertain_low: float, uncertain_high: float) -> pd.DataFrame:
    out = df_pred.copy()
    out['Ensemble_prob'] = ensemble_prob(out['Transformer_prob'].to_numpy(), out['XGBoost_prob'].to_numpy(), wt=wt, wx=wx)

    reg = out['Regime'].astype(str).str.upper()
    out['Model Used'] = np.where(reg == 'HIGH', 'Transformer', np.where(reg == 'LOW', 'XGBoost', 'Ensemble'))
    out['Probability'] = np.where(
        out['Model Used'] == 'Transformer', out['Transformer_prob'].to_numpy(),
        np.where(out['Model Used'] == 'XGBoost', out['XGBoost_prob'].to_numpy(), out['Ensemble_prob'].to_numpy())
    )

    out['binary_pred'] = (out['Probability'] >= threshold).astype(int)
    out['Confidence'] = np.where((out['Probability'] >= uncertain_low) & (out['Probability'] <= uncertain_high), 'Low', 'High')
    out['Prediction'] = np.where(out['Confidence'] == 'Low', 'UNCERTAIN', np.where(out['binary_pred'] == 1, 'UP', 'DOWN'))
    return out


final_df = apply_final_logic(
    df_pred=aligned_test,
    threshold=balanced_threshold,
    wt=w_transformer,
    wx=w_xgb,
    uncertain_low=UNCERTAIN_LOW,
    uncertain_high=UNCERTAIN_HIGH,
)

display(final_df[['Ticker', 'Date', 'Regime', 'Prediction', 'Probability', 'Confidence', 'Model Used']].head(12))

,Ticker,Date,Regime,Prediction,Probability,Confidence,Model Used
0,ABB,2025-01-29,HIGH,UNCERTAIN,0.494498,Low,Transformer
1,ABB,2025-01-30,HIGH,UNCERTAIN,0.502987,Low,Transformer
2,ABB,2025-01-31,HIGH,UNCERTAIN,0.492936,Low,Transformer
3,ABB,2025-02-01,HIGH,UNCERTAIN,0.506073,Low,Transformer
4,ABB,2025-02-03,HIGH,UNCERTAIN,0.496291,Low,Transformer
5,ABB,2025-02-04,HIGH,UNCERTAIN,0.490754,Low,Transformer
6,ABB,2025-02-05,HIGH,UNCERTAIN,0.494325,Low,Transformer
7,ABB,2025-02-06,HIGH,UNCERTAIN,0.499879,Low,Transformer
8,ABB,2025-02-07,HIGH,UNCERTAIN,0.497488,Low,Transformer
9,ABB,2025-02-10,HIGH,UNCERTAIN,0.502407,Low,Transformer


## STEP 6: Final Output Table

In [8]:
final_output = final_df[['Ticker', 'Date', 'Regime', 'Prediction', 'Probability', 'Confidence', 'Model Used']].copy()
final_output = final_output.sort_values(['Ticker', 'Date']).reset_index(drop=True)
display(final_output.head(10))

,Ticker,Date,Regime,Prediction,Probability,Confidence,Model Used
0,ABB,2025-01-29,HIGH,UNCERTAIN,0.494498,Low,Transformer
1,ABB,2025-01-30,HIGH,UNCERTAIN,0.502987,Low,Transformer
2,ABB,2025-01-31,HIGH,UNCERTAIN,0.492936,Low,Transformer
3,ABB,2025-02-01,HIGH,UNCERTAIN,0.506073,Low,Transformer
4,ABB,2025-02-03,HIGH,UNCERTAIN,0.496291,Low,Transformer
5,ABB,2025-02-04,HIGH,UNCERTAIN,0.490754,Low,Transformer
6,ABB,2025-02-05,HIGH,UNCERTAIN,0.494325,Low,Transformer
7,ABB,2025-02-06,HIGH,UNCERTAIN,0.499879,Low,Transformer
8,ABB,2025-02-07,HIGH,UNCERTAIN,0.497488,Low,Transformer
9,ABB,2025-02-10,HIGH,UNCERTAIN,0.502407,Low,Transformer


## STEP 7: Final Evaluation (Baseline Transformer vs Improved Pipeline)

In [9]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    return {
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'Recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'F1': float(f1_score(y_true, y_pred, zero_division=0)),
        'ROC_AUC': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        'FP': int(((y_true == 0) & (y_pred == 1)).sum()),
        'FN': int(((y_true == 1) & (y_pred == 0)).sum()),
    }


y_true = final_df['target'].to_numpy()
baseline_pred = final_df['Transformer_pred'].to_numpy()
baseline_prob = final_df['Transformer_prob'].to_numpy()

pipeline_pred = final_df['binary_pred'].to_numpy()
pipeline_prob = final_df['Probability'].to_numpy()

baseline_row = evaluate(y_true, baseline_pred, baseline_prob)
pipeline_row = evaluate(y_true, pipeline_pred, pipeline_prob)

comparison = pd.DataFrame([
    {'Model': 'Baseline Transformer', **baseline_row},
    {'Model': 'Improved Pipeline', **pipeline_row},
])
display(comparison[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'FP', 'FN']].round(4))

uncertain_rate = float((final_df['Prediction'] == 'UNCERTAIN').mean())
print(f'Uncertain rate: {uncertain_rate:.2%}')

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,FP,FN
0,Baseline Transformer,0.5175,0.5181,0.5379,0.5278,0.5200,5490,5071
1,Improved Pipeline,0.5053,0.5067,0.5032,0.5049,0.5068,5377,5452


Uncertain rate: 40.89%


## STEP 8: Save Final Output

In [10]:
PATHS['final_predictions'].parent.mkdir(parents=True, exist_ok=True)
final_output.to_parquet(PATHS['final_predictions'], index=False)
logger.info('Saved final output: %s', PATHS['final_predictions'])

[INFO] Saved final output: c:\Users\Priyanshu\Desktop\Main\Financial-Marketing-Forecasting\ml_pipeline\Market_Data\final\final_predictions.parquet


## Summary

In [11]:
base = comparison.set_index('Model').loc['Baseline Transformer']
impr = comparison.set_index('Model').loc['Improved Pipeline']

print('Final pipeline completed.')
print(f'Prior threshold: {prior_threshold:.3f} -> Balanced threshold: {balanced_threshold:.3f}')
print(f"F1 delta: {impr['F1'] - base['F1']:+.4f}")
print(f"Precision delta: {impr['Precision'] - base['Precision']:+.4f}")
print(f"Recall delta: {impr['Recall'] - base['Recall']:+.4f}")
print(f"FP delta: {int(impr['FP'] - base['FP']):+d} | FN delta: {int(impr['FN'] - base['FN']):+d}")
print(f'Confidence band [{UNCERTAIN_LOW:.2f}, {UNCERTAIN_HIGH:.2f}] uncertain rate: {uncertain_rate:.2%}')

Final pipeline completed.
Prior threshold: 0.410 -> Balanced threshold: 0.500
F1 delta: -0.0229
Precision delta: -0.0115
Recall delta: -0.0347
FP delta: -113 | FN delta: +381
Confidence band [0.48, 0.52] uncertain rate: 40.89%
